# Data Preparation — CloudTrail Identity Logs
## Federated HGNN Prototype — BINUS Thesis 2026

## Step 1: Load Raw CloudTrail Logs

In [1]:
# !pip install pandas

In [14]:
import pandas as pd
import json
import glob
import os

# path to local CloudTrail folder
cloudtrail_folder = os.path.expanduser("~/Downloads/federated-hgnn-anomaly-detection/Dataset/Raw/CloudTrail/CloudTrail/*.json")

records = []

print("Loading CloudTrail logs...")

for file in glob.glob(cloudtrail_folder):
    with open(file, "r") as f:
        data = json.load(f)
        if "Records" in data:
            records.extend(data["Records"])

print("Total raw records:", len(records))

df_identity = pd.DataFrame(records)

# Limit to 1000 rows for prototyping
df_identity = df_identity.head(1000)

print("Shape:", df_identity.shape)

Loading CloudTrail logs...
Total raw records: 2900
Shape: (1000, 27)


In [15]:
print("Number of records loaded:", len(records))
print()

if len(records) > 0:
    print("Keys in first record:")
    for key in records[0].keys():
        print(" -", key)
else:
    print("Records list is EMPTY — check your CloudTrail folder path")
    print("Folder being searched:", cloudtrail_folder)
    print()
    import glob, os
    files_found = glob.glob(cloudtrail_folder)
    print("JSON files found:", len(files_found))
    for f in files_found:
        print(" -", f)

Number of records loaded: 2900

Keys in first record:
 - eventVersion
 - userIdentity
 - eventTime
 - eventSource
 - eventName
 - awsRegion
 - sourceIPAddress
 - userAgent
 - requestParameters
 - responseElements
 - requestID
 - eventID
 - readOnly
 - eventType
 - managementEvent
 - recipientAccountId
 - eventCategory
 - tlsDetails


## Step 2: Initial Inspection

In [16]:
df_identity.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 27 columns):
 #   Column                        Non-Null Count  Dtype 
---  ------                        --------------  ----- 
 0   eventVersion                  1000 non-null   str   
 1   userIdentity                  1000 non-null   object
 2   eventTime                     1000 non-null   str   
 3   eventSource                   1000 non-null   str   
 4   eventName                     1000 non-null   str   
 5   awsRegion                     1000 non-null   str   
 6   sourceIPAddress               1000 non-null   str   
 7   userAgent                     1000 non-null   str   
 8   requestParameters             927 non-null    object
 9   responseElements              142 non-null    object
 10  requestID                     999 non-null    str   
 11  eventID                       1000 non-null   str   
 12  readOnly                      1000 non-null   bool  
 13  eventType                     

In [28]:
df_identity.head()

,eventVersion,eventTime,eventSource,eventName,awsRegion,sourceIPAddress,userAgent,requestParameters,responseElements,requestID,eventID,readOnly,eventType,managementEvent,recipientAccountId,eventCategory,tlsDetails,resources,errorCode,errorMessage,additionalEventData,sharedEventID,apiVersion,vpcEndpointId,sessionCredentialFromConsole,serviceEventDetails,type,principalId,arn,accountId,accessKeyId,userName,invokedBy,sessionContext.attributes.creationDate,sessionContext.attributes.mfaAuthenticated,sessionContext.sessionIssuer.type,sessionContext.sessionIssuer.principalId,sessionContext.sessionIssuer.arn,sessionContext.sessionIssuer.accountId,sessionContext.sessionIssuer.userName,sessionContext.ec2RoleDelivery,event_hour,event_dayofweek,event_date,is_weekend,is_off_hours,is_anomaly
0,1.08,2023-07-10 12:06:32+00:00,iam.amazonaws.com,CreateRole,us-east-1,192.168.10.20,APN/1.0 HashiCorp/1.0 Terraform/1.1.2 (+https://www.terraform.io) terraform-provider-aws/4.67.0 (+https://registry.terraform.io/providers/hashicorp/aws) aws-sdk-go/1.44.261 (go1.19.8; linux; amd64) stratus-red-team_bc31c885-5ea0-4a6e-8bec-b6b10058bc44 HashiCorp-terraform-exec/0.17.3,"{'path': '/', 'roleName': 'stratus-red-team-ec2lui-role-pcccexdthk', 'assumeRolePolicyDocument': '{""Statement"":[{""Action"":[""sts:AssumeRole"",""sts:SetSourceIdentity""],""Effect"":""Allow"",""Principal"":{""AWS"":""123837392027""},""Sid"":""AllowAssumeRole""}],""Version"":""2012-10-17""}', 'maxSessionDuration': 3600, 'tags': [{'key': 'StratusRedTeam', 'value': 'true'}]}","{'role': {'assumeRolePolicyDocument': '%7B%22Statement%22%3A%5B%7B%22Action%22%3A%5B%22sts%3AAssumeRole%22%2C%22sts%3ASetSourceIdentity%22%5D%2C%22Effect%22%3A%22Allow%22%2C%22Principal%22%3A%7B%22AWS%22%3A%22123837392027%22%7D%2C%22Sid%22%3A%22AllowAssumeRole%22%7D%5D%2C%22Version%22%3A%222012-10-17%22%7D', 'arn': 'arn:aws:iam::123837392027:role/stratus-red-team-ec2lui-role-pcccexdthk', 'roleId': 'AROATFQR7NSC4RHD6IN2N', 'createDate': 'Jul 10, 2023 12:06:32 PM', 'roleName': 'stratus-red-team-ec2lui-role-pcccexdthk', 'path': '/', 'tags': [{'key': 'StratusRedTeam', 'value': 'true'}]}}",f0edd1d5-4012-422b-9cb8-8d2abcf77311,5ae76ecb-47d9-4c92-8065-ebe0f8292243,False,AwsApiCall,True,123837392027,Management,"{'tlsVersion': 'TLSv1.2', 'cipherSuite': 'ECDHE-RSA-AES128-GCM-SHA256', 'clientProvidedHostHeader': 'iam.amazonaws.com'}",NaN,NoError,NoError,NaN,NaN,NaN,NaN,NaN,NaN,IAMUser,AIDATFQR7NSC5AU2ZV3IE,arn:aws:iam::123837392027:user/bert-jan,123837392027,AKIATFQR7NSC8Q4X20BJ,bert-jan,Unknown,NaN,Unknown,Unknown,NaN,NaN,NaN,Unknown,NaN,12,0,2023-07-10,0,0,0
1,1.08,2023-07-10 12:06:32+00:00,iam.amazonaws.com,GetRole,us-east-1,192.168.10.20,APN/1.0 HashiCorp/1.0 Terraform/1.1.2 (+https://www.terraform.io) terraform-provider-aws/4.67.0 (+https://registry.terraform.io/providers/hashicorp/aws) aws-sdk-go/1.44.261 (go1.19.8; linux; amd64) stratus-red-team_bc31c885-5ea0-4a6e-8bec-b6b10058bc44 HashiCorp-terraform-exec/0.17.3,{'roleName': 'stratus-red-team-ec2lui-role-pcccexdthk'},None,34fde66a-cf70-4de7-94c6-461dad3aea30,bfb909d2-000b-4b3c-977f-36ec74c5701c,True,AwsApiCall,True,123837392027,Management,"{'tlsVersion': 'TLSv1.2', 'cipherSuite': 'ECDHE-RSA-AES128-GCM-SHA256', 'clientProvidedHostHeader': 'iam.amazonaws.com'}",NaN,NoError,NoError,NaN,NaN,NaN,NaN,NaN,NaN,IAMUser,AIDATFQR7NSC5AU2ZV3IE,arn:aws:iam::123837392027:user/bert-jan,123837392027,AKIATFQR7NSC8Q4X20BJ,bert-jan,Unknown,NaN,Unknown,Unknown,NaN,NaN,NaN,Unknown,NaN,12,0,2023-07-10,0,0,0
2,1.08,2023-07-10 12:06:32+00:00,iam.amazonaws.com,ListRolePolicies,us-east-1,192.168.10.20,APN/1.0 HashiCorp/1.0 Terraform/1.1.2 (+https://www.terraform.io) terraform-provider-aws/4.67.0 (+https://registry.terraform.io/providers/hashicorp/aws) aws-sdk-go/1.44.261 (go1.19.8; linux; amd64) stratus-red-team_bc31c885-5ea0-4a6e-8bec-b6b10058bc44 HashiCorp-terraform-exec/0.17.3,{'roleName': 'stratus-red-team-ec2lui-role-pcccexdthk'},None,cc6ef3ca-3a64-4a96-94b5-4699c548d4d1,f56f75a6-1118-4a10-83

In [29]:
df_identity.tail()

,eventVersion,eventTime,eventSource,eventName,awsRegion,sourceIPAddress,userAgent,requestParameters,responseElements,requestID,eventID,readOnly,eventType,managementEvent,recipientAccountId,eventCategory,tlsDetails,resources,errorCode,errorMessage,additionalEventData,sharedEventID,apiVersion,vpcEndpointId,sessionCredentialFromConsole,serviceEventDetails,type,principalId,arn,accountId,accessKeyId,userName,invokedBy,sessionContext.attributes.creationDate,sessionContext.attributes.mfaAuthenticated,sessionContext.sessionIssuer.type,sessionContext.sessionIssuer.principalId,sessionContext.sessionIssuer.arn,sessionContext.sessionIssuer.accountId,sessionContext.sessionIssuer.userName,sessionContext.ec2RoleDelivery,event_hour,event_dayofweek,event_date,is_weekend,is_off_hours,is_anomaly
995,1.08,2023-07-10 11:57:49+00:00,secretsmanager.amazonaws.com,DescribeSecret,us-east-1,192.168.10.20,APN/1.0 HashiCorp/1.0 Terraform/1.1.2 (+https://www.terraform.io) terraform-provider-aws/3.76.1 (+https://registry.terraform.io/providers/hashicorp/aws) aws-sdk-go/1.44.157 (go1.19.3; linux; amd64) stratus-red-team_561fe49b-e6b4-44da-a587-f2c718eb578a HashiCorp-terraform-exec/0.17.3,{'secretId': 'arn:aws:secretsmanager:us-east-1:123837392027:secret:stratus-red-team-retrieve-secret-9-7ChiHt'},None,afeb4594-a3af-4c09-a74d-c3c69887a105,43b0757c-3ffd-4528-8f3a-b80ceac9b2a7,True,AwsApiCall,True,123837392027,Management,"{'tlsVersion': 'TLSv1.3', 'cipherSuite': 'TLS_AES_128_GCM_SHA256', 'clientProvidedHostHeader': 'secretsmanager.us-east-1.amazonaws.com'}",NaN,NoError,NoError,NaN,NaN,NaN,NaN,NaN,NaN,IAMUser,AIDATFQR7NSC5AU2ZV3IE,arn:aws:iam::123837392027:user/bert-jan,123837392027,AKIATFQR7NSC8Q4X20BJ,bert-jan,Unknown,NaN,Unknown,Unknown,NaN,NaN,NaN,Unknown,NaN,11,0,2023-07-10,0,0,0
996,1.08,2023-07-10 11:57:48+00:00,secretsmanager.amazonaws.com,DescribeSecret,us-east-1,192.168.10.20,APN/1.0 HashiCorp/1.0 Terraform/1.1.2 (+https://www.terraform.io) terraform-provider-aws/3.76.1 (+https://registry.terraform.io/providers/hashicorp/aws) aws-sdk-go/1.44.157 (go1.19.3; linux; amd64) stratus-red-team_561fe49b-e6b4-44da-a587-f2c718eb578a HashiCorp-terraform-exec/0.17.3,{'secretId': 'arn:aws:secretsmanager:us-east-1:123837392027:secret:stratus-red-team-retrieve-secret-1-ywtiPr'},None,06db582a-9f87-4236-925a-da603118bdb3,c967cb92-90b5-4978-a582-226bf462272a,True,AwsApiCall,True,123837392027,Management,"{'tlsVersion': 'TLSv1.3', 'cipherSuite': 'TLS_AES_128_GCM_SHA256', 'clientProvidedHostHeader': 'secretsmanager.us-east-1.amazonaws.com'}",NaN,NoError,NoError,NaN,NaN,NaN,NaN,NaN,NaN,IAMUser,AIDATFQR7NSC5AU2ZV3IE,arn:aws:iam::123837392027:user/bert-jan,123837392027,AKIATFQR7NSC8Q4X20BJ,bert-jan,Unknown,NaN,Unknown,Unknown,NaN,NaN,NaN,Unknown,NaN,11,0,2023-07-10,0,0,0
997,1.08,2023-07-10 11:57:49+00:00,secretsmanager.amazonaws.com,PutSecretValue,us-east-1,192.168.10.20,APN/1.0 HashiCorp/1.0 Terraform/1.1.2 (+https://www.terraform.io) terraform-provider-aws/3.76.1 (+https://registry.terraform.io/providers/hashicorp/aws) aws-sdk-go/1.44.157 (go1.19.3; linux; amd64) stratus-red-team_561fe49b-e6b4-44da-a587-f2c718eb578a HashiCorp-terraform-exec/0.17.3,"{'secretId': 'arn:aws:secretsmanager:us-east-1:123837392027:secret:stratus-red-team-retrieve-secret-7-nFvpuv', 'clientRequestToken': 'EFADA32A-134C-40A6-880A-6C1421FFD45F'}",{'arn': 'arn:aws:secretsmanager:us-east-1:123837392027:secret:stratus-red-team-retrieve-secret-7-nFvpuv'},5d6b65b7-8f1e-4ac9-8476-d288dcc5e963,1b07449e-fb57-4fde-8fd7-e52f65ab368e,False,AwsApiCall,True,123837392027,Management,"{'tlsVersion': 'TLSv1.3', 'cipherSuite': 'TLS_AES_128_GCM_SHA256', 'clientProvidedHostHeader': 'secretsmanager.us-east-1.amazonaws.com'}",NaN,NoError,NoError,NaN,NaN,NaN,NaN,NaN,NaN,IAMUser,AIDATFQR7NSC5AU2ZV3IE,arn:aws:iam::123837392027:user/bert-jan,123837392027,AKIATFQR7NSC8Q4X20BJ,bert-jan,Unknown,NaN,Unknown,Unknown,NaN,NaN,NaN,Unknown,NaN,11,0,2023-07-10,0,0,0
998,1.08,2023-07-10 11:57:49+00:00,secretsmanager.amazonaws.com,GetRes

### Note on Missing Values
High missing % in columns like `errorCode`, `responseElements`, `vpcEndpointId`, and `additionalEventData` is **expected and normal** in CloudTrail.
These fields only appear when a specific event condition is triggered (e.g., `errorCode` only exists if the API call failed).
This is **structurally sparse data by design**, not a data quality problem.

## Step 3: Flatten userIdentity (Nested JSON Column)

In [17]:
# userIdentity is a nested dict — must be flattened before any ML use
user_identity_df = pd.json_normalize(df_identity['userIdentity'])

print("Flattened userIdentity columns:")
print(user_identity_df.columns.tolist())

# Merge flattened columns back into main dataframe
df_identity = pd.concat([df_identity.drop(columns=['userIdentity']), user_identity_df], axis=1)

print("\nNew shape after flattening:", df_identity.shape)

Flattened userIdentity columns:
['type', 'principalId', 'arn', 'accountId', 'accessKeyId', 'userName', 'invokedBy', 'sessionContext.attributes.creationDate', 'sessionContext.attributes.mfaAuthenticated', 'sessionContext.sessionIssuer.type', 'sessionContext.sessionIssuer.principalId', 'sessionContext.sessionIssuer.arn', 'sessionContext.sessionIssuer.accountId', 'sessionContext.sessionIssuer.userName', 'sessionContext.ec2RoleDelivery']

New shape after flattening: (1000, 41)


## Step 4: Handle Missing Values

In [18]:
# Compute missing value report BEFORE any fixes
missing = df_identity.isnull().sum().sort_values(ascending=False)
missing_percent = (missing / len(df_identity)) * 100
missing_table = pd.concat([missing, missing_percent], axis=1)
missing_table.columns = ["Missing Count", "Missing %"]

# Show only columns that actually have missing values
print("Columns with missing values (before fix):")
missing_table[missing_table["Missing Count"] > 0].head(20)

Columns with missing values (before fix):


,Missing Count,Missing %
serviceEventDetails,999,99.9
apiVersion,999,99.9
sessionContext.ec2RoleDelivery,990,99.0
sharedEventID,986,98.6
sessionContext.sessionIssuer.userName,943,94.3
sessionContext.sessionIssuer.accountId,943,94.3
sessionContext.sessionIssuer.arn,943,94.3
sessionContext.sessionIssuer.principalId,943,94.3
sessionContext.sessionIssuer.type,943,94.3
vpcEndpointId,939,93.9


In [19]:
# These are the identity columns needed for the HGNN graph nodes and edges.
# Missing here = ambiguous identity = treat as 'Unknown' (preserves the signal).
identity_cols = [
    'type',
    'principalId',
    'accountId',
    'accessKeyId',
    'userName',
    'invokedBy',
    'sessionContext.attributes.mfaAuthenticated',
    'sessionContext.sessionIssuer.type',
    'sessionContext.sessionIssuer.userName'
]

# Only apply fillna to columns that actually exist (safety check)
existing_identity_cols = [c for c in identity_cols if c in df_identity.columns]
df_identity[existing_identity_cols] = df_identity[existing_identity_cols].fillna("Unknown")

# Verify fix was applied
print("Missing values in identity columns AFTER fix:")
print(df_identity[existing_identity_cols].isnull().sum())

Missing values in identity columns AFTER fix:
type                                          0
principalId                                   0
accountId                                     0
accessKeyId                                   0
userName                                      0
invokedBy                                     0
sessionContext.attributes.mfaAuthenticated    0
sessionContext.sessionIssuer.type             0
sessionContext.sessionIssuer.userName         0
dtype: int64


In [20]:
# For error-related columns, NaN means 'no error occurred' — fill with 'NoError'
error_cols = ['errorCode', 'errorMessage']
for col in error_cols:
    if col in df_identity.columns:
        df_identity[col] = df_identity[col].fillna("NoError")

print("Error columns filled. Sample:")
print(df_identity['errorCode'].value_counts().head())

Error columns filled. Sample:
errorCode
NoError                         882
Client.UnauthorizedOperation     44
ThrottlingException              26
AccessDenied                      7
NoSuchBucketPolicy                5
Name: count, dtype: int64


## Step 5: Extract Time Features from eventTime

In [21]:
df_identity['eventTime'] = pd.to_datetime(df_identity['eventTime'], utc=True)

df_identity['event_hour']      = df_identity['eventTime'].dt.hour
df_identity['event_dayofweek'] = df_identity['eventTime'].dt.dayofweek
df_identity['event_date']      = df_identity['eventTime'].dt.date
df_identity['is_weekend']      = df_identity['event_dayofweek'].isin([5, 6]).astype(int)

# is_off_hours: access outside 08:00-18:00 = potential anomaly signal
df_identity['is_off_hours'] = (~df_identity['event_hour'].between(8, 18)).astype(int)

print("Time features added.")
df_identity[['eventTime', 'event_hour', 'event_dayofweek', 'is_weekend', 'is_off_hours']].head()

Time features added.


,eventTime,event_hour,event_dayofweek,is_weekend,is_off_hours
0,2023-07-10 12:06:32+00:00,12,0,0,0
1,2023-07-10 12:06:32+00:00,12,0,0,0
2,2023-07-10 12:06:32+00:00,12,0,0,0
3,2023-07-10 12:06:32+00:00,12,0,0,0
4,2023-07-10 12:06:35+00:00,12,0,0,0


## Step 6: Create Anomaly Labels
This is a **rule-based heuristic label** for prototype purposes.
An event is flagged as anomalous if it matches known attacker patterns in the Invictus-IR dataset.

In [22]:
# Known high-risk API calls associated with privilege escalation and recon
# Source: Invictus-IR attack scenario documentation
HIGH_RISK_EVENTS = [
    'AssumeRole',
    'CreateAccessKey',
    'AttachUserPolicy',
    'AttachRolePolicy',
    'PutUserPolicy',
    'CreateLoginProfile',
    'UpdateLoginProfile',
    'GetSecretValue',
    'ListBuckets',
    'GetObject',
]

# Rule: flag as anomaly if high-risk event AND (off-hours OR no MFA OR there is an errorCode that is not NoError)
is_high_risk_event = df_identity['eventName'].isin(HIGH_RISK_EVENTS)
is_no_mfa = df_identity.get('sessionContext.attributes.mfaAuthenticated', pd.Series(['Unknown']*len(df_identity))) != 'true'
is_failed = df_identity['errorCode'] != 'NoError'

df_identity['is_anomaly'] = (
    is_high_risk_event & (df_identity['is_off_hours'] == 1)
    | is_high_risk_event & is_no_mfa
    | is_high_risk_event & is_failed
).astype(int)

print("Label distribution:")
print(df_identity['is_anomaly'].value_counts())
print(f"\nAnomaly rate: {df_identity['is_anomaly'].mean():.2%}")

Label distribution:
is_anomaly
0    936
1     64
Name: count, dtype: int64

Anomaly rate: 6.40%


## Step 7: Select Final Columns for HGNN Graph Construction
These columns map directly to the graph schema defined in the thesis methodology:
- **Nodes**: `userName` (identity), `sourceIPAddress` (network bridge), `eventSource` (target service)
- **Edges**: defined by `eventName` (what action was taken)
- **Features**: time features, MFA status, error status
- **Label**: `is_anomaly`

In [23]:
FINAL_COLS = [
    # Core identifiers (graph nodes)
    'userName',
    'sourceIPAddress',
    'eventSource',
    'eventName',
    # Identity metadata
    'type',
    'accessKeyId',
    'sessionContext.attributes.mfaAuthenticated',
    'sessionContext.sessionIssuer.type',
    # Event metadata
    'awsRegion',
    'eventType',
    'readOnly',
    'errorCode',
    # Time features
    'event_hour',
    'event_dayofweek',
    'is_weekend',
    'is_off_hours',
    # Label
    'is_anomaly'
]

# Only keep columns that exist in the dataframe
final_cols_existing = [c for c in FINAL_COLS if c in df_identity.columns]
df_final = df_identity[final_cols_existing].copy()

print("Final dataframe shape:", df_final.shape)
print("\nRemaining missing values:")
print(df_final.isnull().sum()[df_final.isnull().sum() > 0])

df_final.head()

Final dataframe shape: (1000, 17)

Remaining missing values:
Series([], dtype: int64)


,userName,sourceIPAddress,eventSource,eventName,type,accessKeyId,sessionContext.attributes.mfaAuthenticated,sessionContext.sessionIssuer.type,awsRegion,eventType,readOnly,errorCode,event_hour,event_dayofweek,is_weekend,is_off_hours,is_anomaly
0,bert-jan,192.168.10.20,iam.amazonaws.com,CreateRole,IAMUser,AKIATFQR7NSC8Q4X20BJ,Unknown,Unknown,us-east-1,AwsApiCall,False,NoError,12,0,0,0,0
1,bert-jan,192.168.10.20,iam.amazonaws.com,GetRole,IAMUser,AKIATFQR7NSC8Q4X20BJ,Unknown,Unknown,us-east-1,AwsApiCall,True,NoError,12,0,0,0,0
2,bert-jan,192.168.10.20,iam.amazonaws.com,ListRolePolicies,IAMUser,AKIATFQR7NSC8Q4X20BJ,Unknown,Unknown,us-east-1,AwsApiCall,True,NoError,12,0,0,0,0
3,bert-jan,192.168.10.20,iam.amazonaws.com,ListAttachedRolePolicies,IAMUser,AKIATFQR7NSC8Q4X20BJ,Unknown,Unknown,us-east-1,AwsApiCall,True,NoError,12,0,0,0,0
4,bert-jan,192.168.10.20,ec2.amazonaws.com,AssociateRouteTable,IAMUser,AKIATFQR7NSC8Q4X20BJ,Unknown,Unknown,us-east-1,AwsApiCall,False,NoError,12,0,0,0,0


In [ ]:
# Save the cleaned dataframe for the next stage (Graph Construction)
output_path = os.path.expanduser("~/Downloads/federated-hgnn-anomaly-detection/Dataset/Processed/df_identity_clean.csv")
df_final.to_csv(output_path, index=False)
print(f"Saved to: {output_path}")
print("Data preparation complete. Ready for graph construction.")

Saved to: /Users/philberttan/Downloads/Pre Thesis/Dataset/Processed/df_identity_clean.csv
Data preparation complete. Ready for graph construction.
